# DP-MLM: Reproduction and Evaluation on New Data

This notebook implements a DP-MLM-style text rewriting pipeline, evaluates utility on SST-2, tests generalisation on IMDb and AG News, constructs a synthetic PII dataset, measures attribute retention, and runs an LLM-as-judge evaluation.

**Experiment structure**
1. **Reproduction component:** evaluate the DP-MLM utility pipeline on SST-2, an original GLUE task.
2. **New-data evaluation:** evaluate the same rewriting and utility pipeline on IMDb and AG News, then evaluate sensitive-attribute retention on synthetic PII sentences.
3. **Supplementary evaluation:** use an instruction-tuned model to score semantic preservation and fluency on rewritten IMDb and AG News examples.

**Reproducibility:** Run all cells from top to bottom in a fresh Google Colab GPU runtime. The SST-2 cell calibrates clipping bounds using training data only, before any privacy rewriting. All experiments use one shared privacy-budget list, `EPSILONS = [10, 25, 50, 100, 250]`.

**Scope note:** The SST-2 experiment reproduces the selected utility-evaluation component; it does not reproduce every experiment in the original paper (for example, the Trustpilot/Yelp adversarial privacy evaluations).


In [ ]:
!pip -q install transformers datasets accelerate sentence-transformers sacrebleu scikit-learn scipy sentencepiece protobuf tiktoken

In [ ]:
import os, random, re, warnings, numpy as np, pandas as pd, torch, sacrebleu
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"; os.environ["WANDB_DISABLED"] = "true"
from tqdm.auto import tqdm
from scipy.stats import spearmanr
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
from transformers import (AutoTokenizer, AutoModelForMaskedLM, AutoModelForSequenceClassification,
                          AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import load_dataset, Dataset
import transformers; print("transformers", transformers.__version__, "| torch", torch.__version__)

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("WARNING: no GPU found. Runtime -> Change runtime type -> T4 GPU, then run again (CPU would take many hours).")
os.makedirs("results", exist_ok=True)

EPSILONS = [10, 25, 50, 100, 250]  # single shared privacy-budget grid for all experiments
N_TRAIN, N_TEST_NEW, MAX_TOK = 2000, 500, 48
CLIP_MIN, CLIP_MAX = None, None   # calibrated from SST-2 training data before privatization
SAMPLES = {}                       # (dataset, eps) -> list of (original, privatised) for LLM judge
RUN_OFFICIAL = False               # set True to run the optional authors' package check after SST-2 replication
OFFCMP = None; N_OFF = 100

## 1. DP-MLM implementation

The implementation uses RoBERTa-base as a masked language model. At each token position, it masks the current token, conditions on the original sentence and the partially rewritten sentence, clips the vocabulary logits, and samples a replacement using an exponential-mechanism-style softmax.

**Privacy-budget interpretation:** The code applies the supplied epsilon value at each token-replacement step. Because a sentence is processed through multiple token steps, the per-step epsilon must not automatically be described as the total sentence-level privacy guarantee. A formal sentence-level guarantee requires accounting for sequential composition across the token steps and stating the relevant adjacency and sensitivity assumptions.

In [ ]:
mlm_tok = AutoTokenizer.from_pretrained("roberta-base")
mlm = AutoModelForMaskedLM.from_pretrained("roberta-base").to(device).eval()
if device == "cuda": mlm.half()
special_ids = torch.tensor(mlm_tok.all_special_ids, device=device)

@torch.no_grad()
def dpmlm_batch(texts, eps, max_tok=MAX_TOK, clip=None):
    enc = mlm_tok(texts, add_special_tokens=False, truncation=True, max_length=max_tok,
                  padding=True, return_tensors="pt")
    orig, am = enc["input_ids"].to(device), enc["attention_mask"].to(device)
    B, L = orig.shape
    if L == 0: return [t for t in texts]
    priv = orig.clone()
    cls = torch.full((B,1), mlm_tok.cls_token_id, dtype=orig.dtype, device=device)
    sep = torch.full((B,1), mlm_tok.sep_token_id, dtype=orig.dtype, device=device)
    one = torch.ones((B,1), dtype=am.dtype, device=device)
    attn = torch.cat([one, am, one, one, am, one], 1)
    bounds = clip if clip is not None else (CLIP_MIN, CLIP_MAX)
    if bounds[0] is None or bounds[1] is None:
        raise RuntimeError("Clipping bounds are not calibrated. Run the SST-2 replication cell first.")
    lo, hi = bounds; delta = hi - lo
    for i in range(L):
        active = am[:, i].bool()
        if not active.any(): break
        cur = priv.clone(); cur[:, i] = mlm_tok.mask_token_id
        inp = torch.cat([cls, orig, sep, sep, cur, sep], 1)
        pos = 1 + L + 2 + i                        # position of the mask in the private segment
        logits = mlm(input_ids=inp, attention_mask=attn).logits[:, pos, :].float().clamp(lo, hi)
        logits[:, special_ids] = -1e4              # never sample special tokens
        probs = torch.softmax(eps * logits / (2 * delta), dim=-1)
        new = torch.multinomial(probs, 1).squeeze(1)
        priv[:, i] = torch.where(active, new, priv[:, i])
    return [mlm_tok.decode(p[a.bool()], skip_special_tokens=True).strip() for p, a in zip(priv, am)]

def privatize(texts, eps, bs=64):
    texts = list(texts)
    order = np.argsort([len(t) for t in texts]); out = [None]*len(texts)
    for s in tqdm(range(0, len(texts), bs), desc=f"DP-MLM eps={eps}", leave=False):
        idx = order[s:s+bs]
        for j, r in zip(idx, dpmlm_batch([texts[j] for j in idx], eps)): out[int(j)] = r
    return out

@torch.no_grad()
def calibrate_bounds(texts, n=200):
    # data-driven clipping bounds: (mean, mean + 4*std) of MLM logits at a random masked position per calibration sentence
    vals = []; rr = random.Random(SEED)
    for t in texts[:n]:
        enc = mlm_tok(t, return_tensors="pt", truncation=True, max_length=MAX_TOK).to(device)
        L = enc["input_ids"].shape[1]
        if L < 3: continue
        i = rr.randrange(1, L-1); x = enc["input_ids"].clone(); x[0, i] = mlm_tok.mask_token_id
        lg = mlm(input_ids=x, attention_mask=enc["attention_mask"]).logits[0, i].float()
        keep = torch.ones(lg.numel(), dtype=torch.bool, device=device); keep[special_ids] = False
        vals.append(lg[keep].cpu())
    if not vals:
        raise ValueError("Could not collect calibration logits; check that calibration texts contain at least 3 tokens.")
    v = torch.cat(vals); return float(v.mean()), float(v.mean() + 4 * v.std())

def truncate(t):  # baseline text is truncated the same way as privatised text
    return mlm_tok.decode(mlm_tok(t, add_special_tokens=False, truncation=True, max_length=MAX_TOK)["input_ids"]).strip()



## 2. Utility pipeline (DeBERTa-v3-base, BLEU, cosine similarity)

In [ ]:
from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
clf_name = "microsoft/deberta-v3-base"
try:
    clf_tok = AutoTokenizer.from_pretrained(clf_name)
except Exception as ex:                       # fallback if the fast-tokenizer conversion fails
    print("fast tokenizer failed (", repr(ex)[:80], ") -> using slow tokenizer")
    clf_tok = AutoTokenizer.from_pretrained(clf_name, use_fast=False)

def finetune_eval(tr_x, tr_y, te_x, te_y, num_labels, epochs=3):
    set_seed()
    model = AutoModelForSequenceClassification.from_pretrained(
        clf_name, num_labels=num_labels
    ).to(device).float()  # Keep classifier weights in FP32 for stable training.
    mk = lambda x, y: Dataset.from_dict({"text": list(x), "label": [int(v) for v in y]}).map(
        lambda b: clf_tok(b["text"], truncation=True, max_length=96), batched=True)
    tr, te = mk(tr_x, tr_y), mk(te_x, te_y)
    args = TrainingArguments(output_dir="tmp", num_train_epochs=epochs, per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=4,
        learning_rate=3e-5,
        save_strategy="no",
        report_to="none",
        fp16=False,  # Avoid GradScaler FP16 unscale errors.
        bf16=False,
        logging_steps=500,
        seed=SEED)
    def metrics(p):
        pred = p.predictions.argmax(-1)
        return {"acc": accuracy_score(p.label_ids, pred), "f1": f1_score(p.label_ids, pred, average="macro")}
    t = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=te,
                data_collator=DataCollatorWithPadding(clf_tok), compute_metrics=metrics)
    t.train(); r = t.evaluate()
    del model, t; torch.cuda.empty_cache()
    return r["eval_acc"], r["eval_f1"]

def cos_sim(a, b):
    ea = st_model.encode(list(a), normalize_embeddings=True, batch_size=128)
    eb = st_model.encode(list(b), normalize_embeddings=True, batch_size=128)
    return float((ea * eb).sum(1).mean())

def run_utility(name, tr_x, tr_y, te_x, te_y, nl, eps_list):
    tr_x = [truncate(t) for t in tr_x]; te_x = [truncate(t) for t in te_x]
    rows = []
    acc, f1 = finetune_eval(tr_x, tr_y, te_x, te_y, nl)
    rows.append(dict(dataset=name, eps="original", acc=acc, f1=f1, bleu=100.0, cos=1.0))
    print(rows[-1])
    for eps in eps_list:
        ptr, pte = privatize(tr_x, eps), privatize(te_x, eps)
        acc, f1 = finetune_eval(ptr, tr_y, pte, te_y, nl)
        rows.append(dict(dataset=name, eps=eps, acc=acc, f1=f1,
                         bleu=sacrebleu.corpus_bleu(pte, [te_x]).score, cos=cos_sim(te_x, pte)))
        SAMPLES[(name, eps)] = list(zip(te_x, pte))
        print(rows[-1])
        pd.DataFrame([{"orig": o, "priv": p} for o, p in SAMPLES[(name, eps)]]).to_csv(f"results/samples_{name}_eps{eps}.csv", index=False)
    df = pd.DataFrame(rows); df.to_csv(f"results/utility_{name}.csv", index=False); return df

def clean(t):  # remove HTML line breaks (IMDb) and extra whitespace
    return re.sub(r"\s+", " ", re.sub(r"<br\s*/?>", " ", t)).strip()

def sample(ds, n, col="text"):
    ds = ds.shuffle(seed=SEED).select(range(min(n, len(ds))))
    return [clean(t) for t in ds[col]], [int(v) for v in ds["label"]]

## 3. Replication on SST-2 (original GLUE task)

This section uses the labelled SST-2 training and validation splits. Clipping bounds are calibrated on a fixed sample of SST-2 training texts, then held constant for the SST-2, IMDb, AG News and synthetic-PII experiments. The utility pipeline reports accuracy, macro-F1, BLEU and embedding cosine similarity.

The experiment uses 2,000 training examples and 500 validation examples by default. DeBERTa is fine-tuned separately on original and privatized training text for each epsilon value.

In [ ]:
# Load labelled SST-2 data and calibrate the DP-MLM logit clipping range.
sst2 = load_dataset("nyu-mll/glue", "sst2")

N_SST_TRAIN = 2000
N_SST_TEST = 500

sst_train_x, sst_train_y = sample(
    sst2["train"], N_SST_TRAIN, col="sentence"
)
sst_val_x, sst_val_y = sample(
    sst2["validation"], N_SST_TEST, col="sentence"
)

# Calibrate once using SST-2 training data only; do not use validation/test data.
CLIP_MIN, CLIP_MAX = calibrate_bounds(
    sst_train_x, n=min(200, len(sst_train_x))
)
print(f"Calibrated clipping bounds: min={CLIP_MIN:.4f}, max={CLIP_MAX:.4f}")
print(f"SST-2 sample sizes: train={len(sst_train_x)}, validation={len(sst_val_x)}")

# Save the exact validation texts for the optional official-package comparison.
SST_TE = [truncate(t) for t in sst_val_x]

# Qualitative sanity check after calibration.
_demo = ["The film was a delightful surprise with wonderful acting."]
for eps in [10, 50, 250]:
    print(f"epsilon={eps}: {privatize(_demo, eps)[0]}")

# Full SST-2 utility experiment for all five epsilon values.
res_sst = run_utility(
    "SST2",
    sst_train_x,
    sst_train_y,
    sst_val_x,
    sst_val_y,
    2,          # SST-2 has two sentiment labels
    EPSILONS
)
display(res_sst.round(3))
res_sst.to_csv("results/utility_SST2.csv", index=False)


## 4. Evaluation on new datasets: IMDb and AG News (not used in the original paper)

In [ ]:
imdb = load_dataset("stanfordnlp/imdb")
tr_x, tr_y = sample(imdb["train"], N_TRAIN); te_x, te_y = sample(imdb["test"], N_TEST_NEW)
res_imdb = run_utility("IMDb", tr_x, tr_y, te_x, te_y, 2, EPSILONS)

ag = load_dataset("fancyzhx/ag_news")
tr_x, tr_y = sample(ag["train"], N_TRAIN); te_x, te_y = sample(ag["test"], N_TEST_NEW)
res_ag = run_utility("AGNews", tr_x, tr_y, te_x, te_y, 4, EPSILONS)
pd.concat([res_imdb, res_ag])

## 5. Newly constructed dataset


In [ ]:
SYN_PATH = "pii_dataset.csv"
ATTRS = ["name", "job", "city", "org", "phone", "email"]

def build_synthetic(seed=SEED):
    rng = random.Random(seed)
    first = ["Sarah","James","Priya","Wei","Olivia","Liam","Aisha","Carlos","Emma","Noah","Mei","Arjun","Chloe","Ethan","Fatima"]
    last  = ["Johnson","Smith","Patel","Chen","Williams","Brown","Khan","Garcia","Taylor","Nguyen","Singh","Wilson","Lee","Martin","Clark"]
    cities= ["Sydney","Melbourne","Brisbane","Perth","Adelaide","Canberra","Hobart","Darwin","Newcastle","Geelong"]
    orgs  = ["ABC Consulting","Bluewave Logistics","Northstar Bank","Greenfield Health","Orion Software","Redgum Energy","Summit Legal","Harbour Media"]
    jobs  = ["financial analyst","software engineer","nurse","civil engineer","teacher","lawyer","data scientist","accountant"]
    T = {"name_job":"{name} works as {a} {job}.", "name_city":"{name} lives in {city}.",
         "name_org":"{name} works at {org}.", "all":"{name} works at {org} in {city} as {a} {job}.",
         "contact":"You can reach {name} on {phone} or at {email}."}
    rows = []
    for tname, tmpl in T.items():
        for k in range(60):
            f, l = rng.choice(first), rng.choice(last)
            v = dict(name=f"{f} {l}", job=rng.choice(jobs), city=rng.choice(cities), org=rng.choice(orgs),
                     phone=f"04{rng.randint(10,99)} {rng.randint(100,999)} {rng.randint(100,999)}",
                     email=f"{f.lower()}.{l.lower()}@example.com")
            v["a"] = "an" if v["job"][0].lower() in "aeiou" else "a"      # grammar: "an accountant", "a nurse"
            text = tmpl.format(**v)
            rows.append(dict(id=len(rows), template=tname, text=text,
                             **{a: (v[a] if "{"+a+"}" in tmpl else "") for a in ATTRS}))
    return pd.DataFrame(rows)

if os.path.exists(SYN_PATH):
    syn = pd.read_csv(SYN_PATH, dtype=str, keep_default_na=False)     # empty cells stay "" (not NaN)
    need = {"id", "template", "text"} | set(ATTRS)
    assert need <= set(syn.columns), f"CSV is missing columns: {need - set(syn.columns)}"
    syn["id"] = syn["id"].astype(int)
    print(f"Loaded YOUR uploaded dataset: {SYN_PATH}")
else:
    syn = build_synthetic()
    syn.to_csv(SYN_PATH, index=False)
    print(f"{SYN_PATH} not found -> generated it (same seed). You can download it from the Files panel.")
syn.to_csv("results/synthetic_pii_dataset.csv", index=False)
print(syn.shape); syn.sample(5, random_state=1)[["template","text"]]

In [ ]:
attrs = ["name","city","org","job","phone","email"]

def has(value, text):
    # whole-word, case-insensitive match (so 'nurse' does not match 'nursery', 'Lee' does not match 'sleep')
    return re.search(r"(?<!\w)" + re.escape(value) + r"(?!\w)", text, flags=re.I) is not None

leak_rows, SYN_PRIV = [], {}
for eps in EPSILONS:
    priv = privatize(list(syn.text), eps); SYN_PRIV[eps] = priv
    parr = np.array(priv, dtype=object)
    r = {"eps": eps}
    for a in attrs:
        m = (syn[a] != "").values
        r[a] = float(np.mean([has(v, p) for v, p in zip(syn.loc[m, a], parr[m])]))
    # partial name leakage: first OR last name survives
    m = (syn["name"] != "").values
    r["name_part"] = float(np.mean([any(has(w, p) for w in v.split()) for v, p in zip(syn.loc[m, "name"], parr[m])]))
    leak_rows.append(r)
leak = pd.DataFrame(leak_rows); leak.to_csv("results/synthetic_leakage.csv", index=False)
SAMPLES[("Synthetic", 50)] = list(zip(syn.text, SYN_PRIV[50]))
leak.round(3)

## 6. LLM-as-judge (Qwen2.5-1.5B-Instruct)

The judge evaluates rewritten IMDb and AG News examples for semantic preservation and fluency on a 1–5 scale. The notebook samples up to 20 pairs per dataset per epsilon value (three epsilon values), giving up to 120 judged pairs in total. Cosine similarity is also recorded so its association with the LLM semantic score can be examined.

**Limitation:** This is an automated supplementary evaluation. The notebook does not contain completed human ratings, so it does not support a claim of human–LLM agreement.

In [ ]:

# ===== 6. LLM-as-Judge: IMDb + AG News =====

jname = "Qwen/Qwen2.5-1.5B-Instruct"

jtok = AutoTokenizer.from_pretrained(jname)

jmodel = AutoModelForCausalLM.from_pretrained(jname).to(device).eval()

if device == "cuda":
    jmodel.half()

Q = {
    "semantic": "Rate how well the REWRITTEN sentence preserves the meaning of the ORIGINAL sentence, from 1 (meaning completely lost) to 5 (meaning fully preserved).",
    "fluency": "Rate how grammatical and fluent the REWRITTEN sentence is, from 1 (incoherent) to 5 (perfectly fluent)."
}

@torch.no_grad()
def judge(orig, priv, dim):

    msgs = [
        {
            "role": "system",
            "content": "You are a strict evaluator. Reply with a single digit from 1 to 5 and nothing else."
        },
        {
            "role": "user",
            "content": f"{Q[dim]}\n\nORIGINAL: {orig}\nREWRITTEN: {priv}\n\nScore:"
        }
    ]

    prompt = jtok.apply_chat_template(
        msgs,
        tokenize=False,
        add_generation_prompt=True
    )

    ids = jtok(prompt, return_tensors="pt").to(device)

    out = jmodel.generate(
        **ids,
        max_new_tokens=3,
        do_sample=False,
        pad_token_id=jtok.eos_token_id
    )

    generated = jtok.decode(
        out[0, ids["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    m = re.search(r"[1-5]", generated)

    return int(m.group()) if m else np.nan


# Number of samples per dataset for each epsilon
N_J = 20

jrows = []

# Correct dataset names from the previous utility cell
JUDGE_DATASETS = ["IMDb", "AGNews"]

for dataset_name in JUDGE_DATASETS:

    for eps in EPSILONS:

        key = (dataset_name, eps)

        # Check whether samples exist
        if key not in SAMPLES:
            print(f"Skipping {key}: samples not found.")
            continue

        available_pairs = SAMPLES[key]

        n = min(N_J, len(available_pairs))

        pairs = random.Random(
            SEED + int(eps) + len(dataset_name)
        ).sample(available_pairs, n)

        emb_o = st_model.encode(
            [o for o, p in pairs],
            normalize_embeddings=True
        )

        emb_p = st_model.encode(
            [p for o, p in pairs],
            normalize_embeddings=True
        )

        cosine_scores = (emb_o * emb_p).sum(1)

        print(f"\nEvaluating {dataset_name}, epsilon={eps}, samples={n}")

        for (o, p), c in tqdm(
            zip(pairs, cosine_scores),
            total=n,
            desc=f"{dataset_name} eps={eps}"
        ):

            semantic_score = judge(o, p, "semantic")
            fluency_score = judge(o, p, "fluency")

            jrows.append({
                "dataset": dataset_name,
                "eps": eps,
                "orig": o,
                "priv": p,
                "cos": float(c),
                "semantic": semantic_score,
                "fluency": fluency_score
            })


# Create final results
jdf = pd.DataFrame(jrows)

if jdf.empty:
    raise ValueError(
        "No LLM judge results generated. "
        "Run the IMDb and AGNews utility cells first."
    )

# Save results
jdf.to_csv("results/llm_judge.csv", index=False)

# Dataset-wise results
print("\n===== Dataset-wise LLM Judge Results =====")

print(
    jdf.groupby(["dataset", "eps"])[
        ["semantic", "fluency", "cos"]
    ].mean().round(3)
)

# Combined epsilon-wise results
print("\n===== Combined LLM Judge Results =====")

print(
    jdf.groupby("eps")[
        ["semantic", "fluency", "cos"]
    ].mean().round(3)
)

# Spearman correlation
rho, pval = spearmanr(
    jdf["cos"],
    jdf["semantic"],
    nan_policy="omit"
)

print(
    f"\nSpearman(cosine, LLM semantic) = "
    f"{rho:.3f} (p={pval:.3g})"
)

print("\nLLM Judge evaluation completed successfully!")

del jmodel
torch.cuda.empty_cache()


In [ ]:
# OPTIONAL: export a small blinded annotation sheet for a future human evaluation.
# This cell only creates a blank sheet. It does NOT produce human ratings or agreement statistics.
if "jdf" in globals() and not jdf.empty:
    ann = jdf.sample(min(30, len(jdf)), random_state=SEED)[["orig", "priv"]].reset_index(names="row")
    ann["human_semantic"] = ""
    ann["human_fluency"] = ""
    ann.to_csv("annotation_sheet.csv", index=False)
    print("Created blank annotation_sheet.csv. Human evaluation is incomplete until ratings are entered and analysed.")
else:
    print("LLM judge results are not available; run the LLM-as-judge cell first.")


## 6b. (Optional) Check against the authors' official code (`pip install dpmlm`)

Set `RUN_OFFICIAL = True` in the configuration cell to run this optional check. It uses the SST-2 validation sample created in Section 3. Run the notebook from top to bottom first.

In [ ]:
OFFCMP = None
if RUN_OFFICIAL:
    try:
        import subprocess, sys, gc, nltk
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dpmlm"], check=True)
        for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "words", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
            nltk.download(pkg, quiet=True)          # must be BEFORE importing dpmlm (it reads the stopword list at import time)
        try:                                        # compatibility shim: the package sets a torch>=2.9 attribute at import
            _c = type(torch.backends.cuda.matmul); _o = _c.__setattr__
            def _sa(self, n, v):
                try: _o(self, n, v)
                except AttributeError:
                    if n != "fp32_precision": raise
            _c.__setattr__ = _sa
        except Exception: pass
        from dpmlm import DPMLM
        OM = DPMLM(MODEL="FacebookAI/roberta-base",
                   calibration={"clip_min": CLIP_MIN, "clip_max": CLIP_MAX, "sensitivity": CLIP_MAX - CLIP_MIN})
        sub = [truncate(t) for t in SST_TE[:N_OFF]]
        first = lambda o: o if isinstance(o, str) else str(o[0])     # dpmlm_rewrite returns (text, n_perturbed, n_total)
        rows = []
        for eps in EPSILONS:
            set_seed()
            # matched setting: every word token privatised (STOP=True), earlier replacements used as context (REPLACE=True)
            off = [first(OM.dpmlm_rewrite(x, epsilon=eps, STOP=True, REPLACE=True)) for x in tqdm(sub, desc=f"official eps={eps}", leave=False)]
            # package defaults (stop words and punctuation are kept)
            offd = [first(OM.dpmlm_rewrite(x, epsilon=eps)) for x in tqdm(sub, desc=f"official-default eps={eps}", leave=False)]
            mine = privatize(sub, eps)
            rows.append(dict(eps=eps, cos_official=cos_sim(sub, off), cos_mine=cos_sim(sub, mine), cos_official_default=cos_sim(sub, offd),
                             bleu_official=sacrebleu.corpus_bleu(off, [sub]).score, bleu_mine=sacrebleu.corpus_bleu(mine, [sub]).score,
                             bleu_official_default=sacrebleu.corpus_bleu(offd, [sub]).score))
            print(rows[-1])
        OFFCMP = pd.DataFrame(rows); OFFCMP.to_csv("results/official_vs_mine.csv", index=False)
        del OM
    except Exception as ex:
        OFFCMP = None
        print("Official package could not be run (the rest of the notebook is not affected):", repr(ex)[:300])
    finally:
        torch.set_grad_enabled(True)                # the package disables autograd globally -> restore
        import gc; gc.collect(); torch.cuda.empty_cache()
else:
    print("RUN_OFFICIAL = False -> skipped")
OFFCMP

## 7. Summary tables for the report

In [ ]:

# ===== FINAL RESULTS SUMMARY AND DOWNLOAD =====

import os
import shutil
import pandas as pd

os.makedirs("results", exist_ok=True)

# 1. SST-2 results (only if executed previously)
print("\n========== SST-2 RESULTS ==========")

if "res_sst" in globals():
    print(res_sst.round(3).to_string(index=False))
elif os.path.exists("results/utility_SST2.csv"):
    res_sst = pd.read_csv("results/utility_SST2.csv")
    print(res_sst.round(3).to_string(index=False))
else:
    print("SST-2 results not available. SST-2 experiment was not executed.")

# 2. IMDb and AG News results
print("\n========== IMDb + AG NEWS RESULTS ==========")

if "res_imdb" in globals() and "res_ag" in globals():
    print(
        pd.concat([res_imdb, res_ag])
        .round(3)
        .to_string(index=False)
    )
else:
    print("IMDb or AG News results are missing.")

# 3. Synthetic PII leakage results
print("\n========== SYNTHETIC LEAKAGE RESULTS ==========")

if "leak" in globals():
    print(leak.round(3).to_string(index=False))
else:
    print("Leakage results not available.")

# 4. LLM Judge results
print("\n========== LLM JUDGE RESULTS ==========")

if "jdf" in globals() and not jdf.empty:
    print(
        jdf.groupby(["dataset", "eps"])[
            ["semantic", "fluency", "cos"]
        ].mean().round(2)
    )
else:
    print("LLM Judge results not available.")

# 5. Official comparison
print("\n========== OFFICIAL COMPARISON ==========")

if "OFFCMP" in globals() and OFFCMP is not None:
    print(OFFCMP.round(3).to_string(index=False))
else:
    print("Official comparison skipped or unavailable.")

# 6. Copy annotation sheet if available
if os.path.exists("annotation_sheet.csv"):
    shutil.copy(
        "annotation_sheet.csv",
        "results/annotation_sheet.csv"
    )

# 7. Create ZIP file
if os.path.exists("results.zip"):
    os.remove("results.zip")

shutil.make_archive("results", "zip", "results")

print("\n================================")
print("All available results collected.")
print("Created results.zip successfully!")
print("================================")

# 8. Download results
try:
    from google.colab import files
    files.download("results.zip")
except Exception as ex:
    print("Automatic download unavailable:", ex)
